In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [119]:
# Replace with your correct file paths
batting_pp = pd.read_csv('batting_powerplay.csv')
batting_mid = pd.read_csv('batting_middle.csv')
batting_death = pd.read_csv('batting_death.csv')

bowling_pp = pd.read_csv('bowling_powerplay.csv')
bowling_mid = pd.read_csv('bowling_middle.csv')
bowling_death = pd.read_csv('bowling_death.csv')

auctions = pd.read_csv('auctions_final.csv')
# inflation = pd.read_csv('inflation.csv')


In [120]:
# Batting filters
batting_pp = batting_pp[batting_pp['runs'] >= 30]
batting_mid = batting_mid[batting_mid['runs'] >= 40]
batting_death = batting_death[batting_death['runs'] >= 25]

# Bowling filters
bowling_pp = bowling_pp[bowling_pp['balls'] > 5]
bowling_mid = bowling_mid[bowling_mid['balls'] > 5]
bowling_death = bowling_death[bowling_death['balls'] > 5]


In [121]:
bowling_pp.team.value_counts()

team
Sunrisers Hyderabad            152
Mumbai Indians                 151
Royal Challengers Bengaluru    150
Punjab Kings                   149
Delhi Capitals                 148
Kolkata Knight Riders          139
Rajasthan Royals               122
Chennai Super Kings            110
Lucknow Super Giants            90
Gujarat Titans                  71
Name: count, dtype: int64

In [122]:
def normalize_phase(df, metrics, group_cols=['season']):
    df = df.copy()
    for col in metrics:
        df[f'{col}_z'] = df.groupby(group_cols)[col].transform(lambda x: zscore(x, ddof=1) if len(x) > 1 else 0)
    # Shift to positive scale (min-max after z)
    for col in metrics:
        z_col = f'{col}_z'
        df[z_col] = (df[z_col] - df[z_col].min()) / (df[z_col].max() - df[z_col].min() + 1e-9)
    return df


In [123]:
batting_weights = {
    'powerplay': {'runs_z': 0.8, 'strike_rate_z': 0.5, 'boundary_pct_z': 0.45, 'dot_pct_z': -0.3},
    'middle': {'runs_z': 0.8, 'strike_rate_z': 0.3, 'boundary_pct_z': 0.2, 'dot_pct_z': -0.2},
    'death': {'runs_z': 0.8, 'strike_rate_z': 0.5, 'boundary_pct_z': 0.3, 'dot_pct_z': -0.4}
}

bowling_weights = {
    'powerplay': {'wickets_z': 1.5, 'economy_z': -0.5, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.2},
    'middle': {'wickets_z': 1.5, 'economy_z': -0.8, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.2},
    'death': {'wickets_z': 1.5, 'economy_z': -0.4, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.3}
}


In [124]:
def compute_phase_score(df, weights, phase_name, metrics):
    df = normalize_phase(df, metrics)
    df['impact_score'] = sum(df[col] * w for col, w in weights.items())
    min_score = df['impact_score'].min()
    df['impact_score_pos'] = df['impact_score'] + abs(min_score)
    df['phase'] = phase_name
    return df


In [125]:
bat_pp = compute_phase_score(batting_pp, batting_weights['powerplay'], 'powerplay',
                             ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])
bat_mid = compute_phase_score(batting_mid, batting_weights['middle'], 'middle',
                              ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])
bat_death = compute_phase_score(batting_death, batting_weights['death'], 'death',
                                ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])


In [83]:
bat_pp.to_csv("temp.csv")

In [126]:
bowl_pp = compute_phase_score(
    bowling_pp,
    bowling_weights['powerplay'],
    'powerplay',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)

bowl_mid = compute_phase_score(
    bowling_mid,
    bowling_weights['middle'],
    'middle',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)

bowl_death = compute_phase_score(
    bowling_death,
    bowling_weights['death'],
    'death',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)


In [127]:

inflation = pd.DataFrame({
    "year": [2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025],
    "inflation": [2.1229721486,1.9370183837,1.7455333727,1.6363863999,1.5598002096,1.4923461630,
                  1.4442525530,1.3893723453,1.3261165843,1.2437784508,1.1830861323,1.10879675,1.0495]
})


In [128]:
auctions = pd.read_csv("auctions_final.csv")  # or your unified auction file
auctions = auctions.merge(inflation, left_on="year", right_on="year", how="left")
auctions["adj_price"] = auctions["price"] * auctions["inflation"]


In [87]:
df = bat_pp.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_powerplay.csv", index=False)


In [88]:
df = bat_mid.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_middle.csv", index=False)


In [89]:
df = bat_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_death.csv", index=False)


In [90]:
df = bowl_pp.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_powerplay.csv", index=False)


In [91]:
df = bowl_mid.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_middle.csv", index=False)


In [92]:
df = bowl_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_death.csv", index=False)


In [129]:
def inner_join(dfname, dff):
    df = dff.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="inner")
    
    df["ROI"] = df["impact_score_pos"] / (df["adj_price"]/1e8)  # price in millions
    df.to_csv(f"{dfname}_inner_join.csv", index=False)

In [130]:
bowl_pp.team.value_counts()

team
Sunrisers Hyderabad            152
Mumbai Indians                 151
Royal Challengers Bengaluru    150
Punjab Kings                   149
Delhi Capitals                 148
Kolkata Knight Riders          139
Rajasthan Royals               122
Chennai Super Kings            110
Lucknow Super Giants            90
Gujarat Titans                  71
Name: count, dtype: int64

In [101]:
df = bowl_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="inner")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_death_inner_join.csv", index=False)

In [131]:
for i in [(bowl_pp, "bowl_pp"),
    (bowl_mid, "bowl_mid"),
    (bowl_death, "bowl_death"),
    (bat_death, "bat_death"),
    (bat_mid, "bat_mid"),
    (bat_pp, "bat_pp")]:
    inner_join(i[1], i[0])

In [132]:
bat_mid = pd.read_csv("bat_mid_inner_join.csv")
bowl_mid = pd.read_csv("bowl_mid_inner_join.csv")

# Combine all three phases later; here’s the structure for one:
bat_player_roi = (
    bat_mid.groupby(["season","player","team"])["ROI"]
    .mean()   # or sum() if you want cumulative impact
    .reset_index()
)


In [133]:
franchise_roi = (
    bat_mid.groupby(["season","team"])["ROI"].mean().reset_index()
)
franchise_roi.rename(columns={"ROI":"batting_ROI"}, inplace=True)

bowl_franchise = (
    bowl_mid.groupby(["season","team"])["ROI"].mean().reset_index()
)
bowl_franchise.rename(columns={"ROI":"bowling_ROI"}, inplace=True)

# Combine both sides
team_roi = franchise_roi.merge(bowl_franchise, on=["season","team"], how="outer")
team_roi["overall_ROI"] = team_roi[["batting_ROI","bowling_ROI"]].mean(axis=1)
team_roi.to_csv("franchise_overall_roi.csv", index=False)


In [134]:
# --- File mapping ---
files = {
    "bat_pp": "bat_pp_inner_join.csv",
    "bat_mid": "bat_mid_inner_join.csv",
    "bat_death": "bat_death_inner_join.csv",
    "bowl_pp": "bowl_pp_inner_join.csv",
    "bowl_mid": "bowl_mid_inner_join.csv",
    "bowl_death": "bowl_death_inner_join.csv"
}

# --- Load all dataframes into a dictionary ---
dfs = {k: pd.read_csv(v) for k, v in files.items()}

# --- Helper: aggregate by season + team ---
def team_phase_roi(df):
    return (
        df.groupby(["season", "team"])["ROI"]
        .mean()          # avg ROI per franchise per phase
        .reset_index()
    )

# --- Compute ROI per phase ---
team_rois = {k: team_phase_roi(df) for k, df in dfs.items()}

# --- Merge all six phase ROIs ---
final = (
    team_rois["bat_pp"]
    .merge(team_rois["bat_mid"], on=["season", "team"], how="outer", suffixes=("_bat_pp", "_bat_mid"))
    .merge(team_rois["bat_death"], on=["season", "team"], how="outer")
    .merge(team_rois["bowl_pp"], on=["season", "team"], how="outer", suffixes=("", "_bowl_pp"))
    .merge(team_rois["bowl_mid"], on=["season", "team"], how="outer", suffixes=("", "_bowl_mid"))
    .merge(team_rois["bowl_death"], on=["season", "team"], how="outer", suffixes=("", "_bowl_death"))
)

# --- Rename merged columns for clarity ---
final.columns = [
    "season", "team",
    "ROI_bat_pp", "ROI_bat_mid", "ROI_bat_death",
    "ROI_bowl_pp", "ROI_bowl_mid", "ROI_bowl_death"
]

# --- Calculate aggregate ROIs ---
final["batting_ROI"] = final[["ROI_bat_pp", "ROI_bat_mid", "ROI_bat_death"]].mean(axis=1)
final["bowling_ROI"] = final[["ROI_bowl_pp", "ROI_bowl_mid", "ROI_bowl_death"]].mean(axis=1)
final["overall_ROI"] = final[["batting_ROI", "bowling_ROI"]].mean(axis=1)

# --- Save final outputs ---
final.to_csv("franchise_phasewise_and_overall_ROI.csv", index=False)

print("✅ Aggregated ROI file saved as 'franchise_phasewise_and_overall_ROI.csv'")


✅ Aggregated ROI file saved as 'franchise_phasewise_and_overall_ROI.csv'


In [2]:
files = {
    "bat_pp": "bat_pp_inner_join.csv",
    "bat_mid": "bat_mid_inner_join.csv",
    "bat_death": "bat_death_inner_join.csv",
    "bowl_pp": "bowl_pp_inner_join.csv",
    "bowl_mid": "bowl_mid_inner_join.csv",
    "bowl_death": "bowl_death_inner_join.csv"
}

# --- Load all dataframes into a dictionary ---
dfs = {k: pd.read_csv(v) for k, v in files.items()}


In [4]:
for phase, df in dfs.items():
    df["ROI_z"] = (df["ROI"] - df["ROI"].min()) / (df["ROI"].max() - df["ROI"].min() + 1e-9)
    dfs[phase] = df
    df.to_csv(f"{phase}_with_ROI_z.csv", index=False)

In [5]:
# 1. get keys
bat_keys  = [k for k in dfs.keys() if k.lower().startswith("bat")]
bowl_keys = [k for k in dfs.keys() if k.lower().startswith("bowl")]

# 2. concat batting and bowling (preserve origin)
bat_master  = pd.concat([dfs[k].assign(phase=k) for k in bat_keys],  ignore_index=True)
bowl_master = pd.concat([dfs[k].assign(phase=k) for k in bowl_keys], ignore_index=True)

# 3. find common columns (intersection)
common_cols = list(set(bat_master.columns) & set(bowl_master.columns))

# ensure sensible ordering and keep important cols if present
priority = ["season","player","team","phase","adj_price","roi","z_roi"]
cols = [c for c in priority if c in common_cols] + sorted([c for c in common_cols if c not in priority])

# 4. combined df with only common columns
combined_master = pd.concat([bat_master[cols], bowl_master[cols]], ignore_index=True)

# quick sanity prints
print("bat_master rows:", len(bat_master), "cols:", bat_master.shape[1])
print("bowl_master rows:", len(bowl_master), "cols:", bowl_master.shape[1])
print("combined_master rows:", len(combined_master), "cols kept:", len(cols))

# optionally save
combined_master.to_csv("combined_master.csv", index=False)
bat_master.to_csv("bat_master.csv", index=False)
bowl_master.to_csv("bowl_master.csv", index=False)


bat_master rows: 616 cols: 23
bowl_master rows: 1357 cols: 22
combined_master rows: 1973 cols kept: 15


In [1]:
import pandas as pd

In [2]:
combined_master = pd.read_csv("combined_master.csv")

In [12]:
# aggregate per player first to remove duplicates from different phases,
# then roll up to (team, season)
per_player = (
    combined_master
    .groupby(["team", "season", "player"], as_index=False)
    .agg(
        player_adj_spend=("adj_price", "mean"),
        avg_player_roi=("ROI_z", "mean")   # average ROI per player across phases
    )
)

fr_roi = (
    per_player
    .groupby(["team", "season"], as_index=False)
    .agg(
        avg_roi=("avg_player_roi", "mean"),       # mean of per-player ROI
        total_adj_spend=("player_adj_spend", "sum"),  # sum after collapsing duplicates by player
        player_count=("player", "nunique")        # distinct player count
    )
)

# fr_roi now has one row per (team, season) with correct totals/counts

fr_roi.to_csv("franchise_roi_summary.csv", index=False)

In [5]:
wins = pd.read_csv("matches_won_by_team.csv")
matches = pd.read_csv("matches_played_by_team.csv")

perf = matches.merge(wins, how="left", on=["team","season"])
perf["wins"] = perf["wins"].fillna(0)
perf["win_pct"] = perf["wins"] / perf["matches"]
perf

,season,team,matches,wins,win_pct
0,2021,Rajasthan Royals,14,5,0.357143
1,2016,Gujarat Titans,16,9,0.562500
2,2016,Sunrisers Hyderabad,17,11,0.647059
3,2018,Punjab Kings,14,6,0.428571
4,2023,Sunrisers Hyderabad,14,4,0.285714
...,...,...,...,...,...
108,2018,Sunrisers Hyderabad,17,10,0.588235
109,2014,Royal Challengers Bengaluru,14,5,0.357143
110,2014,Sunrisers Hyderabad,14,6,0.428571
111,2017,Delhi Capitals,14,6,0.428571


In [13]:
import pandas as pd

# ---- 1. Load wide standings table ----
# replace path if needed
wide = pd.read_csv("final_standings.csv")

# Inspect — ensure first column is the team name column
# If the team column has a different name, replace 'team' below with that name.
team_col = wide.columns[0]  # usually 'team' or similar
wide = wide.rename(columns={team_col: "team"})
wide["team"] = wide["team"].str.strip()

# ---- 2. Melt wide -> long: season, team, final_standing ----
season_cols = [c for c in wide.columns if c != "team"]
long = wide.melt(id_vars=["team"], value_vars=season_cols,
                 var_name="season", value_name="final_standing")

# Clean season and final_standing types
# If season columns are like 'X2013' or 'Season 2013', try to extract digits
def normalize_season(s):
    try:
        return int(str(s).strip())
    except:
        import re
        m = re.search(r"(\d{4})", str(s))
        return int(m.group(1)) if m else s

long["season"] = long["season"].apply(normalize_season)
# final_standing might be strings; coerce to numeric (non-numeric -> 0)
long["final_standing"] = pd.to_numeric(long["final_standing"], errors="coerce").fillna(0).astype(int)

# ---- 3. Create made_playoffs column ----
long["made_playoffs"] = long["final_standing"].apply(lambda x: 1 if x in (1,2,3,4) else 0)

# ---- 4. Now merge with fr_roi and perf to build fr_final ----
# fr_roi and perf should already exist in your notebook as DataFrames.
# If not, load them before merging.
# Example:
# fr_roi = pd.read_csv("fr_roi.csv")
# perf = pd.read_csv("perf.csv")

# Merge (left join to preserve all franchises)
fr_final = (
    fr_roi
    .merge(perf, on=["team", "season"], how="left")   # wins, matches, win_pct
    .merge(long[["season","team","final_standing","made_playoffs"]], on=["team","season"], how="left")
)

# ---- 5. Fill missing values ----
fr_final["made_playoffs"] = fr_final["made_playoffs"].fillna(0).astype(int)
fr_final["final_standing"] = fr_final["final_standing"].fillna(0).astype(int)

# Quick check
print("fr_final shape:", fr_final.shape)
print(fr_final[["season","team","final_standing","made_playoffs"]].drop_duplicates().head())

# Optionally save:
fr_final.to_csv("fr_final_with_standings.csv", index=False)


fr_final shape: (113, 10)
   season                 team  final_standing  made_playoffs
0    2013  Chennai Super Kings               2              1
1    2014  Chennai Super Kings               3              1
2    2015  Chennai Super Kings               2              1
3    2018  Chennai Super Kings               1              1
4    2019  Chennai Super Kings               2              1


In [10]:
check = combined_master.groupby(["team", "season"], as_index=False)

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_PATH = Path("/mnt/data/combined_master.csv")
OUT_XLSX = Path("/mnt/data/season_team_breakdown.xlsx")

df = pd.read_csv(INPUT_PATH)

# --- auto-detect columns ---
def find_column(df, keywords):
    cols = df.columns.tolist()
    low = [c.lower() for c in cols]
    for kw in keywords:
        for i,c in enumerate(low):
            if kw in c:
                return cols[i]
    return None

season_col = find_column(df, ['season','year'])
team_col = find_column(df, ['team','club'])
player_col = find_column(df, ['player','name','player_name'])
adj_price_col = find_column(df, ['adj_price','price','value','amount'])

df[adj_price_col] = pd.to_numeric(df[adj_price_col], errors='coerce').fillna(0)

# --- group totals ---
group_totals = (
    df.groupby([season_col, team_col], dropna=False)[adj_price_col]
    .sum()
    .reset_index()
    .rename(columns={adj_price_col: "total_adj_price"})
)

# --- player-level breakdown ---
player_sums = (
    df.groupby([season_col, team_col, player_col], dropna=False)[adj_price_col]
    .sum()
    .reset_index()
    .rename(columns={adj_price_col: "player_adj_price"})
)

# add percentage contribution
merged = player_sums.merge(group_totals, on=[season_col, team_col], how="left")
merged["percentage"] = (merged["player_adj_price"] / merged["total_adj_price"] * 100).fillna(0)

# --- write excel ---
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    group_totals.to_excel(writer, index=False, sheet_name="group_totals")
    merged.to_excel(writer, index=False, sheet_name="player_breakdowns")

OUT_XLSX
